# 🏅 Análise de Medalhas Olímpicas por País (1896-2024)

## 👥 Autores
Carlos Lavor Neto & Alexandro Pantoja - UEA

## 📋 Análise Parte 1
- **1.1**: 3 tabelas ordenadas (Verão, Inverno, Total)
- **1.2**: 3 gráficos Top 50 países
- Dados integrados: 1896-2022 (histórico) + Paris 2024


In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (16, 12)
plt.rcParams['font.size'] = 10

print("✅ Bibliotecas importadas!")

✅ Bibliotecas importadas!


In [32]:
BASE_PATH = Path('..')
RAW_PATH = BASE_PATH / 'raw'
BRONZE_PATH = BASE_PATH / 'bronze'
GOLD_PATH = BASE_PATH / 'gold'
OUTPUTS_PATH = BASE_PATH / 'outputs'

print("✅ Caminhos configurados")

✅ Caminhos configurados


---
## 1. Carregamento e Integração de Dados

In [33]:
# Carregar dados históricos (1896-2022, inclui Tóquio 2020 e Pequim 2022)
df_hist = pd.read_csv(RAW_PATH / 'Olympics_1896_2022' / 'world_olympedia_olympics_game_medal_tally.csv')
print(f"📊 Dados históricos: {len(df_hist):,} registros")
print(f"   Período: {df_hist['year'].min()}-{df_hist['year'].max()}")

# Adicionar coluna season
df_hist['season'] = df_hist['edition'].apply(lambda x: 'Winter' if 'Winter' in str(x) else 'Summer')

# TRATAMENTO ESPECIAL: 1956 Equestrian (Stockholm)
# Os Jogos Equestres de 1956 foram realizados em Estocolmo devido à quarentena australiana
# Vamos consolidar com os Jogos de Melbourne para cada país
equestrian_1956 = df_hist[df_hist['edition'] == '1956 Equestrian'].copy()
if len(equestrian_1956) > 0:
    print(f"\n🐴 Consolidando 1956 Equestrian (Stockholm) com 1956 Summer (Melbourne)...")
    
    # Para cada país nos Jogos Equestres
    for idx, row in equestrian_1956.iterrows():
        noc = row['country_noc']
        # Verificar se o país já tem entrada nos Jogos de Verão de 1956
        summer_1956_idx = df_hist[(df_hist['year'] == 1956) & 
                                   (df_hist['country_noc'] == noc) & 
                                   (df_hist['edition'] == '1956 Summer Olympics')].index
        
        if len(summer_1956_idx) > 0:
            # Somar as medalhas equestres às medalhas dos Jogos de Verão
            df_hist.loc[summer_1956_idx[0], 'gold'] += row['gold']
            df_hist.loc[summer_1956_idx[0], 'silver'] += row['silver']
            df_hist.loc[summer_1956_idx[0], 'bronze'] += row['bronze']
            df_hist.loc[summer_1956_idx[0], 'total'] += row['total']
        else:
            # Converter a entrada equestre para Summer Olympics
            df_hist.loc[idx, 'edition'] = '1956 Summer Olympics'
            df_hist.loc[idx, 'edition_id'] = 14
    
    # Remover as entradas duplicadas (mantendo apenas as consolidadas)
    df_hist = df_hist[df_hist['edition'] != '1956 Equestrian'].reset_index(drop=True)
    print(f"   ✅ Consolidação concluída")

# VERIFICAR E REMOVER DUPLICATAS no histórico
duplicates = df_hist.duplicated(subset=['year', 'country_noc', 'season'], keep='first')
if duplicates.any():
    print(f"\n⚠️  Removendo {duplicates.sum()} registros duplicados do histórico")
    print(f"   Países afetados: {', '.join(df_hist[duplicates]['country_noc'].unique())}")
    df_hist = df_hist[~duplicates].copy()

# Carregar Paris 2024
df_paris = pd.read_csv(RAW_PATH / 'Olympics_Paris2024' / 'medals_total.csv')
print(f"\n📊 Dados Paris 2024: {len(df_paris):,} registros")

# Preparar Paris 2024 para integração
df_paris_prep = df_paris.rename(columns={
    'country_code': 'country_noc',
    'Gold Medal': 'gold',
    'Silver Medal': 'silver',
    'Bronze Medal': 'bronze',
    'Total': 'total'
}).copy()

df_paris_prep['year'] = 2024
df_paris_prep['edition'] = '2024 Summer Olympics'
df_paris_prep['edition_id'] = 999
df_paris_prep['season'] = 'Summer'

# VERIFICAR se Paris 2024 já existe no histórico
existing_2024 = df_hist[df_hist['year'] == 2024]
if len(existing_2024) > 0:
    print(f"\n⚠️  Dados de 2024 já existem no histórico! Removendo {len(existing_2024)} registros...")
    df_hist = df_hist[df_hist['year'] != 2024].copy()

# Integrar: histórico + Paris 2024
df_integrated = pd.concat([df_hist, df_paris_prep], ignore_index=True)

# Verificar duplicatas no resultado final
final_dupes = df_integrated.duplicated(subset=['year', 'country_noc', 'season'], keep='first')
if final_dupes.any():
    print(f"\n⚠️  ERRO: Ainda existem {final_dupes.sum()} duplicatas após integração!")
    print("   Removendo duplicatas...")
    df_integrated = df_integrated[~final_dupes].copy()

# Salvar
df_integrated.to_parquet(BRONZE_PATH / 'medals_integrated_1896_2024.parquet', index=False)

print(f"\n✅ Dados integrados: {len(df_integrated):,} registros")
print(f"   Anos: {df_integrated['year'].min()}-{df_integrated['year'].max()}")
print(f"   Países únicos: {df_integrated['country_noc'].nunique()}")

# Estatísticas gerais
total_summer = df_integrated[df_integrated['season']=='Summer']
total_winter = df_integrated[df_integrated['season']=='Winter']
print(f"\n📈 Estatísticas gerais:")
print(f"   Jogos de Verão: {len(total_summer)} participações de países")
print(f"   Jogos de Inverno: {len(total_winter)} participações de países")
print(f"   Total de medalhas distribuídas: {int(df_integrated['total'].sum()):,}")

📊 Dados históricos: 1,807 registros
   Período: 1896-2022

🐴 Consolidando 1956 Equestrian (Stockholm) com 1956 Summer (Melbourne)...
   ✅ Consolidação concluída

📊 Dados Paris 2024: 92 registros

✅ Dados integrados: 1,893 registros
   Anos: 1896-2024
   Países únicos: 160

📈 Estatísticas gerais:
   Jogos de Verão: 1454 participações de países
   Jogos de Inverno: 439 participações de países
   Total de medalhas distribuídas: 21,697


---
## 2. Análise 1.1 - Tabelas Consolidadas

Gerando 3 tabelas ordenadas por total de medalhas (decrescente).

In [34]:
def create_consolidated_table(df, season_filter=None, name=''):    """Cria tabela consolidada de medalhas"""        # Filtrar por season se necessário    if season_filter:        df_filtered = df[df['season'] == season_filter].copy()    else:        df_filtered = df.copy()        # Agrupar por país    result = df_filtered.groupby('country_noc', as_index=False).agg({        'gold': 'sum',        'silver': 'sum',        'bronze': 'sum',        'total': 'sum'    })        # Ordenar por total (decrescente)    result = result.sort_values('total', ascending=False).reset_index(drop=True)        # Adicionar rank    result.insert(0, 'rank', range(1, len(result) + 1))        return resultdef create_professional_html(df, title, emoji):    """Cria HTML com design profissional e minimalista"""        total_medals = int(df['total'].sum())    total_countries = len(df)    highlight_countries = ['BRA', 'CUB']        html = f'''<!DOCTYPE html><html><head>    <meta charset="utf-8">    <meta name="viewport" content="width=device-width, initial-scale=1.0">    <title>{title}</title>    <style>        * {{ margin: 0; padding: 0; box-sizing: border-box; }}        body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; background: #f5f7fa; padding: 40px 20px; color: #2c3e50; }}        .container {{ max-width: 1400px; margin: 0 auto; background: white; border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.08); overflow: hidden; }}        .header {{ background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%); color: white; padding: 40px; text-align: center; }}        .header h1 {{ font-size: 32px; font-weight: 600; margin-bottom: 8px; letter-spacing: -0.5px; }}        .header .subtitle {{ font-size: 16px; opacity: 0.9; font-weight: 400; }}        .content {{ padding: 30px 40px; }}        .info-bar {{ display: flex; justify-content: space-between; padding: 20px 0; border-bottom: 2px solid #e8eaed; margin-bottom: 20px; flex-wrap: wrap; gap: 15px; }}        .info-item {{ display: flex; align-items: center; gap: 8px; }}        .info-item .label {{ font-size: 13px; color: #5f6368; font-weight: 500; text-transform: uppercase; letter-spacing: 0.5px; }}        .info-item .value {{ font-size: 18px; font-weight: 600; color: #1e3c72; }}        .table-container {{ overflow-x: auto; }}        table {{ width: 100%; border-collapse: separate; border-spacing: 0; font-size: 14px; }}        thead {{ background: #f8f9fa; position: sticky; top: 0; z-index: 10; }}        th {{ padding: 16px 12px; text-align: center; font-weight: 600; color: #202124; border-bottom: 2px solid #e8eaed; font-size: 13px; text-transform: uppercase; letter-spacing: 0.5px; }}        th:first-child {{ text-align: center; width: 80px; }}        th:nth-child(2) {{ text-align: left; padding-left: 24px; }}        tbody tr {{ border-bottom: 1px solid #f0f0f0; transition: background-color 0.15s ease; }}        tbody tr:hover {{ background-color: #f8f9fa; }}        td {{ padding: 14px 12px; text-align: center; color: #5f6368; }}        td:first-child {{ font-weight: 600; color: #202124; text-align: center; }}        td:nth-child(2) {{ text-align: left; padding-left: 24px; font-weight: 600; color: #1e3c72; font-family: 'Courier New', monospace; font-size: 13px; }}        td:last-child {{ font-weight: 700; color: #1e3c72; font-size: 15px; }}        .rank-1 {{ background: linear-gradient(135deg, #fff9e6 0%, #fffbf0 100%); }}        .rank-1 td:first-child {{ color: #f39c12; font-size: 16px; }}        .rank-2 {{ background: linear-gradient(135deg, #f5f5f5 0%, #fafafa 100%); }}        .rank-2 td:first-child {{ color: #95a5a6; font-size: 16px; }}        .rank-3 {{ background: linear-gradient(135deg, #fff3e6 0%, #fff8f0 100%); }}        .rank-3 td:first-child {{ color: #cd7f32; font-size: 16px; }}        .highlight {{ background: linear-gradient(135deg, #e8f4f8 0%, #f0f8fc 100%); border-left: 4px solid #1e88e5; }}        .highlight td:nth-child(2) {{ color: #1565c0; font-weight: 700; }}        .medal-cell {{ font-weight: 600; }}        .footer {{ background: #f8f9fa; padding: 30px 40px; text-align: center; border-top: 1px solid #e8eaed; }}        .footer-content {{ display: flex; justify-content: center; gap: 40px; flex-wrap: wrap; margin-bottom: 15px; }}        .footer-item {{ font-size: 13px; color: #5f6368; }}        .footer-item strong {{ color: #202124; }}        .footer-note {{ font-size: 12px; color: #80868b; margin-top: 10px; }}        @media print {{ body {{ background: white; padding: 0; }} .container {{ box-shadow: none; }} .header {{ background: #1e3c72 !important; -webkit-print-color-adjust: exact; print-color-adjust: exact; }} }}        @media (max-width: 768px) {{ .content {{ padding: 20px 15px; }} .header {{ padding: 30px 20px; }} .header h1 {{ font-size: 24px; }} table {{ font-size: 12px; }} th, td {{ padding: 10px 8px; }} }}    </style></head><body>    <div class="container">        <div class="header">            <h1>{emoji} {title}</h1>            <p class="subtitle">Ranking Oficial de Medalhas Olímpicas</p>        </div>        <div class="content">            <div class="info-bar">                <div class="info-item"><span class="label">Período</span><span class="value">1896 - 2024</span></div>                <div class="info-item"><span class="label">Total de Países</span><span class="value">{total_countries}</span></div>                <div class="info-item"><span class="label">Total de Medalhas</span><span class="value">{total_medals:,}</span></div>            </div>            <div class="table-container"><table><thead><tr>                <th>Rank</th><th>País</th><th>🥇 Ouro</th><th>🥈 Prata</th><th>🥉 Bronze</th><th>Total</th>            </tr></thead><tbody>'''        for _, row in df.iterrows():        rank = int(row['rank'])        noc = row['country_noc']        gold = int(row['gold'])        silver = int(row['silver'])        bronze = int(row['bronze'])        total = int(row['total'])                row_class = ''        if noc in highlight_countries:            row_class = 'highlight'        elif rank == 1:            row_class = 'rank-1'        elif rank == 2:            row_class = 'rank-2'        elif rank == 3:            row_class = 'rank-3'                html += f'<tr class="{row_class}"><td>{rank}</td><td>{noc}</td><td class="medal-cell">{gold:,}</td><td class="medal-cell">{silver:,}</td><td class="medal-cell">{bronze:,}</td><td>{total:,}</td></tr>\n'        html += '''</tbody></table></div></div>        <div class="footer">            <div class="footer-content">                <div class="footer-item"><strong>Fonte:</strong> Olympedia.org (1896-2022) + Olympics.com (Paris 2024)</div>                <div class="footer-item"><strong>Gerado em:</strong> Outubro 2024</div>            </div>            <p class="footer-note">Rankings baseados em dados oficiais. Brasil e Cuba destacados para referência.</p>        </div>    </div></body></html>'''        return htmlprint("="*100)print("GERANDO TABELAS CONSOLIDADAS - RANKING COMPLETO")print("="*100)# 1. JOGOS DE VERÃOprint("\n📋 1.1.1 - MEDALHAS JOGOS DE VERÃO (1896-2024)")df_summer = create_consolidated_table(df_integrated, 'Summer', 'summer')print(f"Total de países com medalhas: {len(df_summer)}")# Salvar CSV e Parquetdf_summer.to_csv(OUTPUTS_PATH / 'tables' / 'medals_summer.csv', index=False)df_summer.to_parquet(GOLD_PATH / 'medals_summer.parquet', index=False)# Salvar tabela HTML profissionalhtml_summer = create_professional_html(df_summer, "Medalhas - Jogos de Verão (1896-2024)", "🌞")with open(OUTPUTS_PATH / 'tables' / 'medals_summer_full.html', 'w', encoding='utf-8') as f:    f.write(html_summer)print("\nTop 10:")print(df_summer.head(10).to_string(index=False))print("\n... (tabela completa disponível)")print(f"\n🔗 Tabela HTML profissional: outputs/tables/medals_summer_full.html ({len(df_summer)} países)")# 2. JOGOS DE INVERNOprint("\n\n📋 1.1.2 - MEDALHAS JOGOS DE INVERNO (1896-2024)")df_winter = create_consolidated_table(df_integrated, 'Winter', 'winter')print(f"Total de países com medalhas: {len(df_winter)}")# Salvar CSV e Parquetdf_winter.to_csv(OUTPUTS_PATH / 'tables' / 'medals_winter.csv', index=False)df_winter.to_parquet(GOLD_PATH / 'medals_winter.parquet', index=False)# Salvar tabela HTML profissionalhtml_winter = create_professional_html(df_winter, "Medalhas - Jogos de Inverno (1896-2024)", "❄️")with open(OUTPUTS_PATH / 'tables' / 'medals_winter_full.html', 'w', encoding='utf-8') as f:    f.write(html_winter)print("\nTop 10:")print(df_winter.head(10).to_string(index=False))print("\n... (tabela completa disponível)")print(f"\n🔗 Tabela HTML profissional: outputs/tables/medals_winter_full.html ({len(df_winter)} países)")# 3. TOTAL GERALprint("\n\n📋 1.1.3 - MEDALHAS TOTAL GERAL (Verão + Inverno, 1896-2024)")df_total = create_consolidated_table(df_integrated, None, 'total')print(f"Total de países com medalhas: {len(df_total)}")# Salvar CSV e Parquetdf_total.to_csv(OUTPUTS_PATH / 'tables' / 'medals_total.csv', index=False)df_total.to_parquet(GOLD_PATH / 'medals_total.parquet', index=False)# Salvar tabela HTML profissionalhtml_total = create_professional_html(df_total, "Medalhas - Total Geral (1896-2024)", "🏆")with open(OUTPUTS_PATH / 'tables' / 'medals_total_full.html', 'w', encoding='utf-8') as f:    f.write(html_total)print("\nTop 10:")print(df_total.head(10).to_string(index=False))print("\n... (tabela completa disponível)")print(f"\n🔗 Tabela HTML profissional: outputs/tables/medals_total_full.html ({len(df_total)} países)")print("\n" + "="*100)print("✅ 3 tabelas profissionais geradas e salvas!")print("   • CSVs: outputs/tables/medals_*.csv")print("   • HTMLs: outputs/tables/medals_*_full.html (DESIGN PROFISSIONAL)")print("   • Parquets: gold/medals_*.parquet")print("   • Estilo: Minimalista com paleta azul corporativa")print("   • Pronto para: Apresentações, impressão, relatórios")print("="*100)

GERANDO TABELAS CONSOLIDADAS - RANKING COMPLETO

📋 1.1.1 - MEDALHAS JOGOS DE VERÃO (1896-2024)
Total de países com medalhas: 159

Top 10:
 rank country_noc  gold  silver  bronze  total
    1         USA  1122     891     792   2805
    2         GBR   314     356     349   1019
    3         URS   395     319     296   1010
    4         FRA   262     292     323    877
    5         GER   255     286     304    845
    6         CHN   303     226     198    727
    7         ITA   241     214     233    688
    8         AUS   180     189     228    597
    9         JPN   189     162     193    544
   10         HUN   190     168     186    544

... (tabela completa disponível)

🔗 Tabela HTML completa: outputs/tables/medals_summer_full.html (159 países)

📊 TABELA COMPLETA - Primeiros 20 e Brasil/Cuba:
 rank country_noc  gold  silver  bronze  total
    1         USA  1122     891     792   2805
    2         GBR   314     356     349   1019
    3         URS   395     319     296   10

---
## 3. Análise 1.2 - Gráficos Top 50

Gerando 3 gráficos de barras horizontais com Top 50 países.

In [35]:
def plot_top50_medals(df, title, filename):    """Gera gráfico de barras verticais profissional - Top 50"""        # Selecionar Top 50 (ou todos se forem menos de 50)    n_countries = min(50, len(df))    df_plot = df.head(n_countries).copy()        # Criar figura com estilo profissional    fig, ax = plt.subplots(figsize=(24, 10))        # Configurar estilo    plt.style.use('seaborn-v0_8-whitegrid')        x_pos = np.arange(len(df_plot))    bar_width = 0.75        # Cores profissionais (azul corporativo)    color_gold = '#FFB300'      # Dourado profissional    color_silver = '#90A4AE'    # Prata profissional    color_bronze = '#8D6E63'    # Bronze profissional        # Barras empilhadas verticais    bars1 = ax.bar(x_pos, df_plot['gold'], bar_width,                   color=color_gold, alpha=0.9, label='🥇 Ouro',                   edgecolor='white', linewidth=0.5)        bars2 = ax.bar(x_pos, df_plot['silver'], bar_width,                   bottom=df_plot['gold'],                   color=color_silver, alpha=0.9, label='🥈 Prata',                   edgecolor='white', linewidth=0.5)        bars3 = ax.bar(x_pos, df_plot['bronze'], bar_width,                   bottom=df_plot['gold'] + df_plot['silver'],                   color=color_bronze, alpha=0.9, label='🥉 Bronze',                   edgecolor='white', linewidth=0.5)        # Configurar eixo X (países)    ax.set_xticks(x_pos)    ax.set_xticklabels(df_plot['country_noc'], rotation=90, ha='center',                       fontsize=8, fontfamily='monospace', fontweight='bold')        # Configurar eixo Y    ax.set_ylabel('Total de Medalhas', fontsize=14, fontweight='bold', labelpad=15)        # Título profissional    ax.set_title(title, fontsize=18, fontweight='bold', pad=25, color='#1e3c72')        # Legenda profissional    ax.legend(loc='upper right', fontsize=12, framealpha=0.95,             edgecolor='#cccccc', fancybox=False, shadow=False)        # Grade sutil    ax.grid(axis='y', alpha=0.2, linestyle='-', linewidth=0.5, color='#cccccc')    ax.set_axisbelow(True)        # Remover bordas superiores e direitas    ax.spines['top'].set_visible(False)    ax.spines['right'].set_visible(False)    ax.spines['left'].set_color('#cccccc')    ax.spines['bottom'].set_color('#cccccc')        # Destacar Top 3 e Brasil/Cuba    highlight_countries = ['BRA', 'CUB']    for i, (idx, row) in enumerate(df_plot.iterrows()):        noc = row['country_noc']        total = int(row['total'])        rank = int(row['rank'])                # Adicionar total no topo da barra        y_pos = total + df_plot['total'].max() * 0.02                # Cor e peso da fonte        if rank <= 3:            color = '#1e3c72'            weight = 'bold'            size = 9        elif noc in highlight_countries:            color = '#1565c0'            weight = 'bold'            size = 9        else:            color = '#5f6368'            weight = 'normal'            size = 8                ax.text(i, y_pos, f'{total:,}', ha='center', va='bottom',               fontsize=size, fontweight=weight, color=color, rotation=0)                # Borda mais forte para países destacados        if noc in highlight_countries:            bars1[i].set_edgecolor('#1565c0')            bars1[i].set_linewidth(2)            bars2[i].set_edgecolor('#1565c0')            bars2[i].set_linewidth(2)            bars3[i].set_edgecolor('#1565c0')            bars3[i].set_linewidth(2)        elif rank <= 3:            bars1[i].set_edgecolor('#1e3c72')            bars1[i].set_linewidth(1.5)            bars2[i].set_edgecolor('#1e3c72')            bars2[i].set_linewidth(1.5)            bars3[i].set_edgecolor('#1e3c72')            bars3[i].set_linewidth(1.5)        # Ajustar layout    plt.tight_layout()        # Salvar com alta qualidade    plt.savefig(OUTPUTS_PATH / 'figures' / filename, dpi=300, bbox_inches='tight',               facecolor='white', edgecolor='none')    plt.close()        print(f"   ✅ {filename}")print("="*100)print("GERANDO GRÁFICOS PROFISSIONAIS - TOP 50 PAÍSES")print("="*100)print("\n📊 Gráfico 1: Jogos de Verão (Top 50)...")plot_top50_medals(df_summer,                   'Top 50 Países - Medalhas em Jogos de Verão (1896-2024)',                   'top50_summer.png')print("\n📊 Gráfico 2: Jogos de Inverno (Top 46 - todos os países)...")plot_top50_medals(df_winter,                   'Ranking Completo - Medalhas em Jogos de Inverno (1896-2024)',                   'top50_winter.png')print("\n📊 Gráfico 3: Total Geral (Top 50)...")plot_top50_medals(df_total,                   'Top 50 Países - Total de Medalhas Olímpicas (1896-2024)',                   'top50_total.png')print("\n" + "="*100)print("✅ 3 gráficos profissionais gerados!")print("   • Formato: Barras verticais empilhadas")print("   • Quantidade: Top 50 países")print("   • Estilo: Profissional com cores corporativas")print("   • Resolução: 300 DPI para impressão")print("="*100)

GERANDO GRÁFICOS PROFISSIONAIS - TOP 100 PAÍSES

📊 Gráfico 1: Jogos de Verão (Top 100)...
   ✅ top50_summer.png

📊 Gráfico 2: Jogos de Inverno (Top 47 - todos os países)...
   ✅ top50_winter.png

📊 Gráfico 3: Total Geral (Top 100)...
   ✅ top50_total.png

✅ 3 gráficos profissionais gerados!
   • Formato: Barras verticais empilhadas
   • Quantidade: Top 100 países (ou todos se menos de 100)
   • Estilo: Profissional com cores corporativas
   • Resolução: 300 DPI para impressão


---
## 4. Metadados e Documentação

In [36]:
# Criar metadados para cada resultado
datasets = [
    ('medals_summer', 'Medalhas - Jogos de Verão (1896-2024)', df_summer),
    ('medals_winter', 'Medalhas - Jogos de Inverno (1896-2024)', df_winter),
    ('medals_total', 'Medalhas - Total Geral (1896-2024)', df_total)
]

for name, desc, df_data in datasets:
    metadata = {
        'nome': desc,
        'descricao': f'Quadro consolidado de medalhas olímpicas ordenado por total',
        'periodo': '1896-2024',
        'total_paises': len(df_data),
        'total_medalhas': int(df_data['total'].sum()),
        'colunas': list(df_data.columns),
        'fonte': 'Olympedia.org (1896-2022) + Olympics.com (Paris 2024)',
        'notas': 'Dados integrados das fontes oficiais, incluindo tratamento especial para 1956 Equestrian',
        'criado_em': datetime.now().isoformat()
    }
    
    with open(GOLD_PATH / f'{name}_metadata.json', 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    
    print(f"✅ {name}_metadata.json")

print("\n✅ Metadados criados!")

✅ medals_summer_metadata.json
✅ medals_winter_metadata.json
✅ medals_total_metadata.json

✅ Metadados criados!
